kernel : Python (Pixi)

#### Bioproject: PRJNA1108209 (Reference Genome: calJac240_pri)

In [ ]:
%load_ext autoreload
%autoreload 2
import gc
import os
import pandas as pd
import scanpy as sc
from pipeline.utils.env import find_env_dir

merged_h5ad_dir = find_env_dir("MERGED_H5AD")
pre_h5ad_dir = find_env_dir("PRE_H5AD")

In [7]:
MERGED_FILE = "PRJNA1108209.h5ad"
SERIES_NAME = "lin"
SAVE_FILE_NAME = SERIES_NAME + "_raw.h5ad"
adata = sc.read_h5ad(os.path.join(merged_h5ad_dir, MERGED_FILE))
adata.obs.head() #type: ignore

,composite_class,composite_consistency,run,source_name,tissue,cell line,cell type,il01 uniqueid,il02 species,il03 source,...,il08 condition,il09 illumina,il10 chemistry,il11 batch,il12 lmindays,il13 lmaxdays,il14 dataset,geo_loc_name,collection_date,sample
GTCGTTCTCTTCGGAA-1-SRR28906683,0,0.260563,SRR28906683,fWM,fWM,tissue,tissue,v3.9.6,cjacchus,CJP03,...,T2.Lesion,NovaS2,10xv3,9th,1369,1383,this.manuscript,missing,missing,GSM8253397
TAGTGCAAGCAGAAAG-1-SRR28906683,0,0.232394,SRR28906683,fWM,fWM,tissue,tissue,v3.9.6,cjacchus,CJP03,...,T2.Lesion,NovaS2,10xv3,9th,1369,1383,this.manuscript,missing,missing,GSM8253397
ATGGGAGAGGGCAGGA-1-SRR28906683,0,0.190141,SRR28906683,fWM,fWM,tissue,tissue,v3.9.6,cjacchus,CJP03,...,T2.Lesion,NovaS2,10xv3,9th,1369,1383,this.manuscript,missing,missing,GSM8253397
CGAGGAACACATAACC-1-SRR28906683,0,0.225352,SRR28906683,fWM,fWM,tissue,tissue,v3.9.6,cjacchus,CJP03,...,T2.Lesion,NovaS2,10xv3,9th,1369,1383,this.manuscript,missing,missing,GSM8253397
GCTGGGTGTAGCCAGA-1-SRR28906683,0,0.225352,SRR28906683,fWM,fWM,tissue,tissue,v3.9.6,cjacchus,CJP03,...,T2.Lesion,NovaS2,10xv3,9th,1369,1383,this.manuscript,missing,missing,GSM8253397


In [43]:
assert isinstance(adata.obs, pd.DataFrame)
adata.obs["run"] = adata.obs["run"]
adata.obs["matter"] = adata.obs["tissue"]
adata.obs["donor"] = adata.obs["il03 source"]
adata.obs["sex"] = adata.obs["il04 sex"]
adata.obs["age"] = adata.obs["il05 agedays"].astype(float)
adata.obs["location"] = adata.obs["il07 location"].str.split('.', n=1).str[1]
adata.obs["batch"] = adata.obs["il11 batch"]
adata.obs["lesion_age_min"] = adata.obs["il12 lmindays"].astype(float)
adata.obs["lesion_age_max"] = adata.obs["il13 lmaxdays"].astype(float)
adata.obs["sample"] = adata.obs["sample"]
adata.obs["series"] = SERIES_NAME

age_mean = adata.obs["age"].mean()
age_sd = adata.obs["age"].std()
adata.obs["age_scale"] = (adata.obs["age"] - age_mean) / (2 * age_sd)

keep = ["run", "matter", "donor", "sex", "age_scale", "location", "batch", "lesion_age_min", "lesion_age_max", "sample", "series"]
adata.obs.drop(columns = [c for c in adata.obs.columns if c not in keep], inplace=True) #type: ignore

In [45]:
adata.obs.index = adata.obs.index.astype(str)
adata.var.index = adata.var.index.astype(str)
adata.write_h5ad(os.path.join(pre_h5ad_dir, SAVE_FILE_NAME))
del adata
gc.collect()

2406